In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from gwpy.timeseries import TimeSeries
from scipy.fft import fft, ifft
plt.rc('font', **{'family': 'serif', 'serif': ['Times']})
plt.rc('text', usetex=False)


/home/ale/miniconda3/envs/gwenv/lib/python3.10/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


In [2]:
def corrcoef(x, y):
    """
    Return Pearson correlation coefficient between two 1D arrays.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    return np.corrcoef(x, y)[0, 1]


In [3]:
def make_correlated_gaussian_noise(n_samples, sigma=1.0, rho=0.5, seed=None):
    """
    Generate two correlated Gaussian noise channels.

    Parameters
    ----------
    n_samples : int
        Number of samples.
    sigma : float
        Standard deviation of each channel (before correlation scaling).
    rho : float
        Desired Pearson correlation coefficient between channels (-1 to 1).
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    x, y : np.ndarray
        Two arrays of shape (n_samples,) with correlation ~ rho.
    """
    rng = np.random.default_rng(seed)

    # independent standard normals
    z1 = rng.normal(0.0, 1.0, n_samples)
    z2 = rng.normal(0.0, 1.0, n_samples)

    # impose correlation
    x = z1
    y = rho * z1 + np.sqrt(1.0 - rho**2) * z2

    # scale to desired sigma
    x = sigma * x
    y = sigma * y

    return x, y


In [4]:
num_files = 100
noisetimes = np.arange(0,1,0.00048828125)
print(len(noisetimes))
np.random.seed(29680801)
mu, sigma = 0,1.0
s = np.random.normal(mu, sigma, 2048*10000)
print(mu-np.mean(s))
amplitude = 1
s1 = s *amplitude
print('Mean of noise sample',np.mean(s1))
#plt.plot(np.arange(0,10000,1), s1)
noise_array_stream1 = []
i=0
for n in np.arange(0, num_files, 1):
    noise = s1[i:i+2048]
   # GaussianNoise_stream1.append(noise)
    noise_array_stream1.append(noise)
    i+=2048
    

print(np.shape(noise_array_stream1))  

2048
-0.00011904831805355545
Mean of noise sample 0.00011904831805355545
(100, 2048)


In [5]:
np.random.seed(23982047) 
mu, sigma = 0,1# set sigma to 1 11.02.2025
s = np.random.normal(mu, sigma, 2048*10000)
amplitude = 1
s1 = s *amplitude
print('Mean of noise sample',np.mean(s1))
noise_array_stream2 = []
i=0
for n in np.arange(0, num_files, 1):
    noise = s1[i:i+2048]
   # GaussianNoise_stream1.append(noise)
    noise_array_stream2.append(noise)
    i+=2048
    
print(np.shape(noise_array_stream2))

Mean of noise sample -7.103928865113932e-05
(100, 2048)


In [6]:
def cross_correlation(data1, data2, times):
    correlations = np.zeros((0,0))
    #cc = np.zeros((np.shape(data1)))
    d1_mean = np.mean(data1)
    d2_mean = np.mean(data2)
    
    corr = np.correlate((data1-d1_mean), (data2-d2_mean), "same")
    correlations = np.append(correlations, corr/len(data1))
   # print(corr)
    nx = len(times) # set signal length time +- 100 samples at 1/2048
    nxs = np.arange(-nx//2, nx//2, 1)

    return correlations, nxs
                  

In [7]:
import math
import random

def cc(x, y):
    correlation = sum((a) * (b) for a, b in zip(x, y)) 
    return correlation

def cross_correlation(data1, data2, times):
    correlations = []
    lags = np.arange(-len(times)/2, len(times)/2, 1)
    nx = len(lags)
    nxs = np.arange(-nx//2, nx//2, 1) #(-nx//2, nx//2, 1)
    for lag in (lags):
         data1_shifted = np.roll(data1, int(lag))
         correlation = cc(data1_shifted, data2)
         correlation = correlation/len(times)
         correlations.append(correlation)
        # if lag == 0 or lag == np.max(lags): print('lag = ', lag, ', correlation = ', correlation); fig, ax1 = plt.subplots(1,1); ax1.plot(times, data1, c='red'); ax1.plot(times, data2, c='blue')
    #print('correlations = ', np.shape(correlations), ', max correlation = ', np.max(np.abs(correlations)))
    return correlations, nxs
        

In [8]:
np.random.seed(19680801)
times = noisetimes
shifts1 = np.random.random((len(times)))
shifts1 = shifts1*(4*10**-3)
shifts2 = np.random.random((len(times)))
shifts2 = shifts2*(4*10**-3)
signals1 = []
signals2 = []
print(np.max(shifts1)*0.4)
print(np.max(shifts2)*0.4)

print(len(times)/2)

0.0015999105958567285
0.0015995203521987557
1024.0


In [ ]:
## NO-INJECTION: build cross-correlation of noise only

signals1 = []
signals2 = []
maxccs = []

for i in np.arange(0, num_files, 1):
    shift1 = shifts1[i]
    shift2 = shifts2[i]
    t0 = np.max(times)/2
    t0 = t0 + shift1

    # mean-center the two noise streams (unchanged)
    noise1 = noise_array_stream1[i] - np.mean(noise_array_stream1[i])
    noise2 = noise_array_stream2[i] - np.mean(noise_array_stream2[i])

    # --- split noise into the same four sections ---
    noise1_sec1 = noise1[0:512]
    noise1_sec2 = noise1[512:1023]
    noise1_sec3 = noise1[1023:1535]
    noise1_sec4 = noise1[1535:2047]

    noise2_sec1 = noise2[0:512]
    noise2_sec2 = noise2[512:1023]
    noise2_sec3 = noise2[1023:1535]
    noise2_sec4 = noise2[1535:2047]

    # --- cross-correlate NOISE-ONLY sections ---
    ccnoise1 = cross_correlation(noise1_sec1, noise2_sec1, times[0:512])
    ccnoise2 = cross_correlation(noise1_sec2, noise2_sec2, times[512:1023])
    ccnoise3 = cross_correlation(noise1_sec3, noise2_sec3, times[1023:1535])
    ccnoise4 = cross_correlation(noise1_sec4, noise2_sec4, times[1535:2047])

    # final correlated stream: ONLY noise correlations (no ccsig)
    final_sig = np.concatenate((ccnoise1[0], ccnoise2[0], ccnoise3[0], ccnoise4[0]))

    # optional, same as before
    plottimes = np.arange(0, len(final_sig), 1)

    maxcc = np.max(final_sig)
    maxccs.append(maxcc)

    # ---- SAVE ----
    np.savetxt('../TranAD_MGB/data/GAUSS/train/'+str(i)+'_GAUSS_Anomaly_Gaussnoise_train.txt', final_sig)



    # labels: all zeros (no injected segment)
    labels = np.zeros((len(final_sig), 1))
    np.save('../TranAD_MGB/data/GAUSS/labels/'+str(i)+'_GAUSS_Anomaly_Gaussnoise_labels.npy', labels)

    print('saved noise-only for', i)


saved noise-only for 0
saved noise-only for 1
saved noise-only for 2
saved noise-only for 3
saved noise-only for 4
saved noise-only for 5
saved noise-only for 6
saved noise-only for 7
saved noise-only for 8
saved noise-only for 9
saved noise-only for 10
saved noise-only for 11
saved noise-only for 12
saved noise-only for 13
saved noise-only for 14
saved noise-only for 15
saved noise-only for 16
saved noise-only for 17
saved noise-only for 18
saved noise-only for 19
saved noise-only for 20
saved noise-only for 21
saved noise-only for 22
saved noise-only for 23
saved noise-only for 24
saved noise-only for 25
saved noise-only for 26
saved noise-only for 27
saved noise-only for 28
saved noise-only for 29
saved noise-only for 30
saved noise-only for 31
saved noise-only for 32
saved noise-only for 33
saved noise-only for 34
saved noise-only for 35
saved noise-only for 36
saved noise-only for 37
saved noise-only for 38
saved noise-only for 39
saved noise-only for 40
saved noise-only for 41
sa